# 01 — Person schema: write an XSD 1.0 schema pythonically

**What you learn:**

- An XSD schema is just XML, so it renders through the shared `XmlRenderer` — no XSD-specific renderer exists.
- Every element declares `ns="xs"`: the renderer composes the prefixed tag `xs:<method-name>` (`sequence` → `xs:sequence`), so the `xs` prefix is declared once per element, never spelled out in a `render_tag`.
- The `xs` prefix is bound as a plain attribute on the root: `schema(xmlns_xs=...)` surfaces as `xmlns:xs="..."`.
- Structure (`element`, `complexType`, `sequence`), a simple element with a built-in type, and an inline restricted `simpleType` (enumeration) all read like the XSD they produce.

**Prerequisites:** None for the XSD dialect.

## 1. Define the schema builder

Subclass `XsdBuilder` and implement `main(self, root)`. Each grammar call composes an `xs:`-prefixed tag from the method name.

In [ ]:
from genro_builders.builder import BuilderHandler
from genro_builders.contrib.xsd import XsdBuilder

XS = "http://www.w3.org/2001/XMLSchema"


class PersonSchema(XsdBuilder):
    """A schema for a <Person> with a name, an age, and a gender enum."""

    def main(self, root):
        schema = root.schema(
            xmlns_xs=XS,
            targetNamespace="urn:example:person",
            elementFormDefault="qualified",
        )

        person = schema.element(name="Person")
        seq = person.complexType().sequence()
        seq.element(name="Name", type="xs:string")
        seq.element(name="Age", type="xs:integer", minOccurs="0")

        gender = seq.element(name="Gender")
        restriction = gender.simpleType().restriction(base="xs:string")
        restriction.enumeration(value="male")
        restriction.enumeration(value="female")
        restriction.enumeration(value="other")

## 2. Mount on the handler

`add_builder` mounts the builder and builds its source tree — no separate `create()` call is needed for a pointer-free schema.

In [ ]:
page = PersonSchema()
BuilderHandler().add_builder(page)
print(page.source.to_xml())

## 3. `render()` — produce the XSD

Rendering rides the core `XmlRenderer`. `doc_header=True` prepends the XML declaration; `pretty=True` indents one node per line. Note how `ns="xs"` produced every `xs:*` tag without a single `render_tag`.

In [ ]:
rendered = page.render(target=False, doc_header=True, pretty=True)
print(rendered)

## 4. It is a real schema

The output is not merely well-formed XML: it is a valid XSD. Parsed with `xmlschema`, it validates conforming documents and rejects violations (an out-of-enum `Gender`).

In [ ]:
import xmlschema

schema = xmlschema.XMLSchema(rendered)
ok = '<Person xmlns="urn:example:person"><Name>Mario</Name><Age>30</Age><Gender>male</Gender></Person>'
bad = '<Person xmlns="urn:example:person"><Name>X</Name><Gender>alien</Gender></Person>'
print("conforming:", schema.is_valid(ok))
print("enum violation:", schema.is_valid(bad))